# V2 Training — Retrain on Cleaned Dataset

Starting from **v1 best.pth**, retrain with:
- V2 dataset (normalized colours + plaid/gradient/chevron patterns + v2 augmentations)
- Adaptive training loop (per-class weighting, dynamic augmentation, label smoothing)
- Discriminative learning rates (deeper layers get smaller LR)
- Async Google Drive checkpoint saving

Same shared Drive folder as v1.  Checkpoints saved as `v2_best.pth` / `v2_latest.pth`.

In [ ]:
import zipfile, os, io, torch, logging as _logging

# ── Configuration ────────────────────────────────────────────────────────────
DRIVE_FOLDER_ID   = "1GwLV_ZFaeSw0w59OU1ToCGuG5EkHgk5f"
DATASET_ZIP_NAME  = "generated_v2.zip"
LOCAL_EXTRACT     = "/content/dataset_v2"
CHECKPOINT_DIR    = "/content/checkpoints"

COLOR_CLASSES = [
    "red", "orange", "yellow", "green", "blue",
    "violet", "purple", "white", "gray", "black",
    "pink", "brown", "olive",
]
NUM_CLASSES = len(COLOR_CLASSES)

# ── Authenticate ─────────────────────────────────────────────────────────────
from google.colab import auth as _colab_auth
_colab_auth.authenticate_user()

import subprocess
subprocess.run(["pip", "install", "-q", "gdown"], check=True)
import gdown
from googleapiclient.discovery import build as _build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

_drive = _build("drive", "v3")
_logging.getLogger("google_auth_httplib2").setLevel(_logging.ERROR)

def _list_folder(folder_id):
    items, page_token = [], None
    while True:
        resp = _drive.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,name,mimeType)",
            pageToken=page_token,
        ).execute()
        items.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return items

def _download_file(file_id, local_path):
    request = _drive.files().get_media(fileId=file_id)
    with open(local_path, "wb") as fh:
        dl = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = dl.next_chunk()

# ── Locate checkpoints folder ────────────────────────────────────────────────
_folder_contents = _list_folder(DRIVE_FOLDER_ID)
_ckpt_folder_entry = next(
    (f for f in _folder_contents
     if f["name"] == "checkpoints"
     and f["mimeType"] == "application/vnd.google-apps.folder"),
    None,
)
if _ckpt_folder_entry is None:
    raise FileNotFoundError("No 'checkpoints' folder found in Drive — run v1 training first.")

DRIVE_CKPT_FOLDER_ID = _ckpt_folder_entry["id"]
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Download checkpoints
_drive_ckpt_files = {f["name"]: f["id"] for f in _list_folder(DRIVE_CKPT_FOLDER_ID)}
for _name, _fid in _drive_ckpt_files.items():
    _local = os.path.join(CHECKPOINT_DIR, _name)
    print(f"Downloading: {_name} ...")
    _download_file(_fid, _local)
print("Checkpoints ready.")

# ── Download & extract V2 dataset ────────────────────────────────────────────
def _find_file(root, name):
    for dirpath, _, filenames in os.walk(root):
        if name in filenames:
            return os.path.join(dirpath, name)
    return None

csv_found = _find_file(LOCAL_EXTRACT, "labels.csv") if os.path.exists(LOCAL_EXTRACT) else None
if csv_found is None:
    _zip_entry = next(
        (f for f in _folder_contents if f["name"] == DATASET_ZIP_NAME), None,
    )
    if _zip_entry is None:
        raise FileNotFoundError(
            f"'{DATASET_ZIP_NAME}' not found in Drive folder!\n"
            f"Upload the v2 dataset zip from dataset_generation.ipynb first."
        )
    print(f"Downloading dataset: {_zip_entry['name']} ...")
    zip_path = "/content/v2_dataset.zip"
    gdown.download(id=_zip_entry["id"], output=zip_path, quiet=False)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(LOCAL_EXTRACT)
    os.remove(zip_path)
    csv_found = _find_file(LOCAL_EXTRACT, "labels.csv")
    if csv_found is None:
        raise FileNotFoundError("labels.csv not found inside the zip!")
    print("Done.")
else:
    print("Dataset already extracted.")

CSV_PATH = csv_found
IMG_DIR  = os.path.join(os.path.dirname(csv_found), "images")
print(f"CSV : {CSV_PATH}")
print(f"Imgs: {IMG_DIR}  ({len(os.listdir(IMG_DIR))} files)")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))

## Dataset & Model

In [ ]:
import random
import pandas as pd
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
import PIL.Image


class ColorDataset(Dataset):
    def __init__(self, csv_path, img_dir, split, color_classes, transform=None):
        df = pd.read_csv(csv_path)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.img_dir = img_dir
        self.color_classes = color_classes
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(os.path.join(self.img_dir, row["filename"]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        label = torch.tensor([row[c] for c in self.color_classes], dtype=torch.float32)
        return img, label


class RandomColorTemperature:
    def __init__(self, strength=0.3):
        self.strength = strength

    def __call__(self, img):
        arr = np.array(img, dtype=np.float32)
        t = random.uniform(-self.strength, self.strength)
        arr[:, :, 0] = np.clip(arr[:, :, 0] * (1.0 + t), 0, 255)
        arr[:, :, 2] = np.clip(arr[:, :, 2] * (1.0 - t), 0, 255)
        return PIL.Image.fromarray(arr.astype(np.uint8))


def create_model(num_classes=13, dropout=0.4):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


def build_transforms(color_jitter_strength=0.6, temp_strength=0.3, erasing_prob=0.5):
    train_tf = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(),
        transforms.Lambda(lambda img: TF.rotate(img, random.choice([0, 90, 180, 270]))),
        transforms.ColorJitter(
            brightness=color_jitter_strength,
            contrast=color_jitter_strength,
            saturation=color_jitter_strength,
            hue=0.07,
        ),
        RandomColorTemperature(strength=temp_strength),
        transforms.ToTensor(),
        transforms.RandomErasing(p=erasing_prob, scale=(0.02, 0.25), ratio=(0.3, 3.3)),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


# ── Load v1 best checkpoint ─────────────────────────────────────────────────
FRESH_START = True   # True = start from v1 best.pth; False = resume from v2_latest.pth

v2_best_path   = os.path.join(CHECKPOINT_DIR, "v2_best.pth")
v2_latest_path = os.path.join(CHECKPOINT_DIR, "v2_latest.pth")
v1_ft_best_path = os.path.join(CHECKPOINT_DIR, "finetune_best.pth")

if not FRESH_START and os.path.exists(v2_latest_path):
    _init_ckpt_path = v2_latest_path
    print(f"Resuming from v2 checkpoint: {_init_ckpt_path}")
elif os.path.exists(v1_ft_best_path):
    _init_ckpt_path = v1_ft_best_path
    print(f"Starting v2 training from v1 finetuned best: {_init_ckpt_path}")
else:
    raise FileNotFoundError("No finetune_best.pth found — run v1 finetuning first.")

model = create_model(num_classes=NUM_CLASSES, dropout=0.4).to(device)
_ckpt = torch.load(_init_ckpt_path, map_location=device)
model.load_state_dict(_ckpt["model_state_dict"])
print(f"Loaded weights from epoch {_ckpt['epoch'] + 1}, val loss {_ckpt['best_val_loss']:.4f}")

# ── DataLoaders ──────────────────────────────────────────────────────────────
BATCH_SIZE = 64
train_tf, val_tf = build_transforms()

train_ds = ColorDataset(CSV_PATH, IMG_DIR, "train", COLOR_CLASSES, train_tf)
val_ds   = ColorDataset(CSV_PATH, IMG_DIR, "val",   COLOR_CLASSES, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

## Pre-Training Error Analysis

Assess v1 model performance on the **v2 dataset** before any training.

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns


def run_error_analysis(model, loader, color_classes, device, label="val"):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            probs = F.softmax(model(imgs), dim=1).cpu()
            all_preds.append(probs)
            all_labels.append(labels)

    preds  = torch.cat(all_preds,  dim=0)
    labels = torch.cat(all_labels, dim=0)
    errors = (preds - labels).abs()
    N, C = preds.shape

    # Per-class MAE
    per_class_mae = errors.mean(dim=0).numpy()
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(color_classes, per_class_mae, color="steelblue")
    ax.set_ylabel("Mean Absolute Error")
    ax.set_title(f"Per-Class MAE ({label} set, N={N})")
    ax.set_ylim(0, max(per_class_mae) * 1.3)
    for bar, val in zip(bars, per_class_mae):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    # Soft confusion matrix
    true_cls = labels.argmax(dim=1)
    confusion = torch.zeros(C, C)
    for c in range(C):
        mask = true_cls == c
        if mask.sum() == 0:
            continue
        confusion[c] = preds[mask].mean(dim=0)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        confusion.numpy(), annot=True, fmt=".3f", cmap="YlOrRd",
        xticklabels=color_classes, yticklabels=color_classes, ax=ax,
        vmin=0, vmax=1,
    )
    ax.set_xlabel("Predicted probability →")
    ax.set_ylabel("True class ↓")
    ax.set_title(f"Soft Confusion Matrix ({label} set)")
    plt.tight_layout()
    plt.show()

    # Top confused pairs
    off_diag = []
    for i in range(C):
        for j in range(C):
            if i != j:
                off_diag.append((color_classes[i], color_classes[j], confusion[i, j].item()))
    off_diag.sort(key=lambda x: -x[2])

    print(f"\nTop 10 confused pairs (true → predicted, avg probability):")
    for true_c, pred_c, val in off_diag[:10]:
        print(f"  {true_c:>8s} → {pred_c:<8s}  {val:.4f}")

    return per_class_mae, confusion


print("── V1 Model on V2 Dataset (before training) ──")
pre_mae, pre_conf = run_error_analysis(model, val_loader, COLOR_CLASSES, device, "val")

## Adaptive Training Loop

Same adaptive strategy as v1 fine-tuning:

| Signal | Controls |
|---|---|
| Per-class val MAE | Class weights (0.5–3.0) |
| Val loss trend | Augmentation strength |
| Stagnation count | Label smoothing |

**Discriminative learning rates**: backbone_early (×0.001), layer3 (×0.01),
layer4 (×0.1), fc (×1.0).  All layers trainable from the start.

In [ ]:
import threading, queue as _queue_mod
import torch.nn.functional as F
from tqdm.notebook import tqdm
from googleapiclient.http import MediaFileUpload

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
V2_MAX_EPOCHS    = 80
V2_EARLY_STOP    = 15
V2_INITIAL_LR    = 5e-5
V2_WEIGHT_DECAY  = 1e-2

LABEL_SMOOTH_MIN = 0.02
LABEL_SMOOTH_MAX = 0.15
AUG_STRENGTH_MIN = 0.3
AUG_STRENGTH_MAX = 0.8

# ══════════════════════════════════════════════════════════════════════════════
#  DISCRIMINATIVE LEARNING RATES
# ══════════════════════════════════════════════════════════════════════════════
def get_param_groups(model, base_lr, weight_decay):
    buckets = {
        "backbone_early": {"params": [], "lr": base_lr * 0.001},
        "layer3":         {"params": [], "lr": base_lr * 0.01},
        "layer4":         {"params": [], "lr": base_lr * 0.1},
        "fc":             {"params": [], "lr": base_lr},
    }
    for name, p in model.named_parameters():
        p.requires_grad = True
        if "fc" in name:
            buckets["fc"]["params"].append(p)
        elif name.startswith("layer4"):
            buckets["layer4"]["params"].append(p)
        elif name.startswith("layer3"):
            buckets["layer3"]["params"].append(p)
        else:
            buckets["backbone_early"]["params"].append(p)

    groups = []
    for bname, b in buckets.items():
        groups.append({"params": b["params"], "lr": b["lr"], "weight_decay": weight_decay})
        n = sum(p.numel() for p in b["params"])
        print(f"  {bname:>16s}: {n:>12,} params, lr={b['lr']:.1e}")
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  {'TOTAL':>16s}: {n_total:>12,} params (all trainable)")
    return groups

# ══════════════════════════════════════════════════════════════════════════════
#  ADAPTIVE STATE
# ══════════════════════════════════════════════════════════════════════════════
class AdaptiveState:
    def __init__(self):
        self.label_smooth   = 0.10
        self.aug_strength   = 0.6
        self.class_weights  = torch.ones(NUM_CLASSES, device=device)

    def recalibrate(self, epoch, val_loss, prev_val_loss,
                    best_val_loss, epochs_no_improve,
                    val_preds, val_labels):
        changes = []

        # 1. Class weights from per-class val MAE
        mae = (val_preds - val_labels).abs().mean(dim=0)
        raw_w = (mae / mae.mean()).clamp(0.5, 3.0)
        self.class_weights = raw_w.to(device)
        top3 = torch.topk(mae, 3)
        top3_str = ", ".join(f"{COLOR_CLASSES[i]}={mae[i]:.4f}" for i in top3.indices)
        changes.append(f"class weights (worst: {top3_str})")

        # 2. Augmentation from val loss trend
        if prev_val_loss is not None:
            val_delta = val_loss - prev_val_loss
            if val_delta > 0.005:
                self.aug_strength = min(AUG_STRENGTH_MAX, self.aug_strength + 0.05)
                changes.append(f"aug ↑ {self.aug_strength:.2f}")
            elif val_delta < -0.005 and self.aug_strength > AUG_STRENGTH_MIN:
                self.aug_strength = max(AUG_STRENGTH_MIN, self.aug_strength - 0.03)
                changes.append(f"aug ↓ {self.aug_strength:.2f}")

        # 3. Label smoothing
        if prev_val_loss is not None:
            if val_loss > best_val_loss + 0.02 and epochs_no_improve >= 2:
                self.label_smooth = min(LABEL_SMOOTH_MAX, self.label_smooth + 0.01)
                changes.append(f"smooth ↑ {self.label_smooth:.2f}")
            elif val_loss <= best_val_loss + 0.005:
                self.label_smooth = max(LABEL_SMOOTH_MIN, self.label_smooth - 0.01)
                changes.append(f"smooth ↓ {self.label_smooth:.2f}")

        if changes:
            print(f"  [adapt] {' | '.join(changes)}")


adapt = AdaptiveState()

# ══════════════════════════════════════════════════════════════════════════════
#  OPTIMIZER
# ══════════════════════════════════════════════════════════════════════════════
param_groups = get_param_groups(model, V2_INITIAL_LR, V2_WEIGHT_DECAY)
optimizer = torch.optim.AdamW(param_groups)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=4, factor=0.5,
)

start_epoch       = 0
best_val_loss     = float("inf")
epochs_no_improve = 0
v2_train_losses   = []
v2_val_losses     = []

if not FRESH_START and os.path.exists(v2_latest_path) and _init_ckpt_path == v2_latest_path:
    v2_ckpt = torch.load(v2_latest_path, map_location=device)
    start_epoch       = v2_ckpt["epoch"] + 1
    best_val_loss     = v2_ckpt["best_val_loss"]
    epochs_no_improve = 0
    v2_train_losses   = v2_ckpt.get("train_losses", [])
    v2_val_losses     = v2_ckpt.get("val_losses", [])
    _as = v2_ckpt.get("adaptive_state", {})
    adapt.label_smooth  = _as.get("label_smooth", adapt.label_smooth)
    adapt.aug_strength  = _as.get("aug_strength", adapt.aug_strength)
    if "class_weights" in _as:
        adapt.class_weights = torch.tensor(_as["class_weights"], device=device)
    if "optimizer_state_dict" in v2_ckpt:
        try:
            optimizer.load_state_dict(v2_ckpt["optimizer_state_dict"])
        except (ValueError, KeyError):
            print("  Optimizer state incompatible — starting fresh optimizer.")
    if "scheduler_state_dict" in v2_ckpt:
        try:
            scheduler.load_state_dict(v2_ckpt["scheduler_state_dict"])
        except (ValueError, KeyError):
            pass
    print(f"Resumed v2 from epoch {start_epoch}, best val loss: {best_val_loss:.4f}")
else:
    print("Starting fresh v2 training (from v1 best, all layers, discriminative LRs).")

# ══════════════════════════════════════════════════════════════════════════════
#  DRIVE UPLOAD WORKER
# ══════════════════════════════════════════════════════════════════════════════
_v2_drive_ckpt_file_ids = dict(_drive_ckpt_files)
_v2_drive_lock = threading.Lock()
_v2_upload_queue = _queue_mod.Queue()

def _to_cpu(obj):
    if isinstance(obj, torch.Tensor):
        return obj.cpu().clone()
    if isinstance(obj, dict):
        return {k: _to_cpu(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu(x) for x in obj]
    return obj

def _v2_upload_worker():
    import google.auth, logging
    from googleapiclient.discovery import build as _build_svc
    logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
    creds, _ = google.auth.default()
    drive_svc = _build_svc("drive", "v3", credentials=creds)
    while True:
        task = _v2_upload_queue.get()
        if task is None:
            _v2_upload_queue.task_done()
            break
        state, path = task
        filename = os.path.basename(path)
        try:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            torch.save(state, path)
            media = MediaFileUpload(path, resumable=True)
            with _v2_drive_lock:
                existing_id = _v2_drive_ckpt_file_ids.get(filename)
            if existing_id:
                drive_svc.files().update(
                    fileId=existing_id, media_body=media,
                ).execute(num_retries=5)
            else:
                created = drive_svc.files().create(
                    body={"name": filename, "parents": [DRIVE_CKPT_FOLDER_ID]},
                    media_body=media, fields="id",
                ).execute(num_retries=5)
                with _v2_drive_lock:
                    _v2_drive_ckpt_file_ids[filename] = created["id"]
            print(f"  [Drive] saved {filename}")
        except Exception as exc:
            print(f"  [Drive] WARNING: {filename}: {exc}")
        finally:
            _v2_upload_queue.task_done()

_v2_worker = threading.Thread(target=_v2_upload_worker, name="V2-DriveUploader", daemon=True)
_v2_worker.start()

# ══════════════════════════════════════════════════════════════════════════════
#  ADAPTIVE TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════
criterion = nn.KLDivLoss(reduction="none")
prev_val_loss     = None
prev_aug_strength = adapt.aug_strength

for epoch in range(start_epoch, V2_MAX_EPOCHS):
    # Rebuild DataLoader if augmentation changed
    if adapt.aug_strength != prev_aug_strength:
        train_tf, _ = build_transforms(
            color_jitter_strength=adapt.aug_strength,
            temp_strength=adapt.aug_strength * 0.5,
            erasing_prob=0.3 + adapt.aug_strength * 0.3,
        )
        train_ds = ColorDataset(CSV_PATH, IMG_DIR, "train", COLOR_CLASSES, train_tf)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=2, pin_memory=True)
        prev_aug_strength = adapt.aug_strength

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    running = 0.0
    uniform = 1.0 / NUM_CLASSES
    for imgs, labels in tqdm(train_loader, desc=f"V2 {epoch+1}/{V2_MAX_EPOCHS} [train]", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        labels = (1 - adapt.label_smooth) * labels + adapt.label_smooth * uniform
        optimizer.zero_grad()
        log_probs = F.log_softmax(model(imgs), dim=1)
        per_sample = criterion(log_probs, labels).sum(dim=1)
        weights = (labels * adapt.class_weights.unsqueeze(0)).sum(dim=1)
        loss = (per_sample * weights).mean()
        loss.backward()
        optimizer.step()
        running += loss.item() * imgs.size(0)
    train_loss = running / len(train_ds)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    running = 0.0
    all_val_preds, all_val_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            log_probs = F.log_softmax(model(imgs), dim=1)
            running += F.kl_div(log_probs, labels, reduction="batchmean").item() * imgs.size(0)
            all_val_preds.append(log_probs.exp().cpu())
            all_val_labels.append(labels.cpu())
    val_loss = running / len(val_ds)
    val_preds  = torch.cat(all_val_preds)
    val_labels = torch.cat(all_val_labels)

    scheduler.step(val_loss)
    v2_train_losses.append(train_loss)
    v2_val_losses.append(val_loss)

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    lr_fc = optimizer.param_groups[-1]["lr"]
    marker = " !" if is_best else ""
    print(f"V2 {epoch+1:>2}/{V2_MAX_EPOCHS} | train: {train_loss:.4f} | val: {val_loss:.4f}"
          f" | lr(fc): {lr_fc:.1e} | no-improve: {epochs_no_improve}"
          f" | smooth: {adapt.label_smooth:.2f} | aug: {adapt.aug_strength:.2f}{marker}")

    # ── Adapt ────────────────────────────────────────────────────────────────
    adapt.recalibrate(
        epoch, val_loss, prev_val_loss,
        best_val_loss, epochs_no_improve,
        val_preds, val_labels,
    )
    prev_val_loss = val_loss

    # ── Checkpoint ───────────────────────────────────────────────────────────
    _v2_upload_queue.join()

    ckpt_data = {
        "epoch": epoch,
        "model_state_dict": _to_cpu(model.state_dict()),
        "optimizer_state_dict": _to_cpu(optimizer.state_dict()),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "epochs_no_improve": epochs_no_improve,
        "train_losses": v2_train_losses.copy(),
        "val_losses": v2_val_losses.copy(),
        "adaptive_state": {
            "label_smooth": adapt.label_smooth,
            "aug_strength": adapt.aug_strength,
            "class_weights": adapt.class_weights.cpu().tolist(),
        },
    }
    _v2_upload_queue.put((ckpt_data, v2_latest_path))
    if is_best:
        _v2_upload_queue.put((ckpt_data, v2_best_path))

    # ── Early stopping ───────────────────────────────────────────────────────
    if epochs_no_improve >= V2_EARLY_STOP:
        print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {V2_EARLY_STOP} epochs).")
        break

# Clean up
_v2_upload_queue.join()
_v2_upload_queue.put(None)
_v2_worker.join()
print(f"\nV2 training complete. Best val loss: {best_val_loss:.4f}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

if not v2_train_losses:
    ckpt = torch.load(v2_latest_path, map_location="cpu")
    v2_train_losses = ckpt.get("train_losses", [])
    v2_val_losses   = ckpt.get("val_losses", [])

epochs_range = range(1, len(v2_train_losses) + 1)

plt.figure(figsize=(10, 4))
plt.plot(epochs_range, v2_train_losses, label="Train Loss")
plt.plot(epochs_range, v2_val_losses,   label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("KL Divergence Loss")
plt.title("V2 Training: Training & Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best val loss: {min(v2_val_losses):.4f} at epoch {v2_val_losses.index(min(v2_val_losses)) + 1}")

## Post-Training Error Analysis

Load `v2_best.pth` and compare with the pre-training baseline.

In [ ]:
if os.path.exists(v2_best_path):
    v2_best = torch.load(v2_best_path, map_location=device)
    model.load_state_dict(v2_best["model_state_dict"])
    print(f"Loaded v2_best.pth (epoch {v2_best['epoch'] + 1}, val loss {v2_best['best_val_loss']:.4f})")
else:
    print("No v2_best.pth found — using model from end of training loop.")

print("\n── Post-Training Val Set Analysis ──")
post_mae, post_conf = run_error_analysis(model, val_loader, COLOR_CLASSES, device, "val")

# Compare pre vs post
print("\n── MAE Improvement (pre → post) ──")
for i, c in enumerate(COLOR_CLASSES):
    delta = post_mae[i] - pre_mae[i]
    arrow = "↓" if delta < 0 else "↑"
    print(f"  {c:>8s}: {pre_mae[i]:.4f} → {post_mae[i]:.4f}  ({arrow} {abs(delta):.4f})")